# Task 3: Heart Disease Prediction
### DevelopersHub Corporation -- AI/ML Engineering Internship

---

## Problem Statement
Heart disease is one of the leading causes of death worldwide. Early prediction of heart disease risk based on patient health data can help doctors take preventive action. In this task, we build a machine learning classification model that predicts whether a patient is at risk of heart disease.

## Objective
- Load and clean the Heart Disease UCI dataset
- Perform Exploratory Data Analysis (EDA) to understand health trends
- Train Logistic Regression and Decision Tree classification models
- Evaluate models using accuracy, ROC-AUC curve, and confusion matrix
- Identify the most important features affecting heart disease prediction

## Dataset
- Source: Heart Disease UCI Dataset (Cleveland Database) loaded directly from GitHub
- 303 patient records, 13 features + 1 target variable
- Target: 0 = No Heart Disease, 1 = Heart Disease Present

## Features Description
| Feature | Description |
|---------|-------------|
| age | Age of the patient |
| sex | Sex (1 = male, 0 = female) |
| cp | Chest pain type (0-3) |
| trestbps | Resting blood pressure (mm Hg) |
| chol | Serum cholesterol (mg/dl) |
| fbs | Fasting blood sugar > 120 mg/dl (1 = true) |
| restecg | Resting ECG results (0-2) |
| thalach | Maximum heart rate achieved |
| exang | Exercise induced angina (1 = yes) |
| oldpeak | ST depression induced by exercise |
| slope | Slope of peak exercise ST segment |
| ca | Number of major vessels colored by fluoroscopy |
| thal | Thalassemia type |
| target | 0 = No disease, 1 = Disease present |

---

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid')

print('All libraries imported successfully.')

---
## Step 2: Load the Dataset

In [ ]:
# Dataset loaded directly from GitHub -- no manual download needed
url = 'https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv'

df = pd.read_csv(url)

print('Dataset loaded successfully.')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')

---
## Step 3: Data Inspection

In [ ]:
print('First 10 rows:')
df.head(10)

In [ ]:
print(f'Shape: {df.shape}')
print()
print('Column Names:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col}')

In [ ]:
df.info()

In [ ]:
print('Descriptive Statistics:')
df.describe().round(2)

---
## Step 4: Data Cleaning

In [ ]:
# Check missing values
print('Missing Values per Column:')
print('-' * 35)
missing = df.isnull().sum()
for col, count in missing.items():
    status = 'No missing values' if count == 0 else f'{count} missing'
    print(f'  {col:12s}: {status}')
print(f'\nTotal missing values: {missing.sum()}')

In [ ]:
# Check duplicates
duplicates = df.duplicated().sum()
print(f'Duplicate rows found: {duplicates}')
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f'Duplicates removed. New shape: {df.shape}')
else:
    print('No duplicates found. Dataset is clean.')

In [ ]:
# Ensure target column exists and is binary
if 'target' not in df.columns:
    df.rename(columns={df.columns[-1]: 'target'}, inplace=True)

df['target'] = (df['target'] > 0).astype(int)

counts = df['target'].value_counts()
print('Target value counts:')
print(f'  0 (No Heart Disease): {counts.get(0, 0)} patients')
print(f'  1 (Heart Disease)   : {counts.get(1, 0)} patients')

---
## Step 5: Exploratory Data Analysis (EDA)

In [ ]:
# =====================================================
#  PLOT 1: Target Variable Distribution
# =====================================================
labels  = ['No Heart Disease', 'Heart Disease']
colors  = ['#3498DB', '#E74C3C']
counts  = df['target'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart
bars = axes[0].bar(labels, counts.values, color=colors, edgecolor='white', width=0.5)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{int(bar.get_height())} patients',
                 ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Patient Count by Disease Status', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Patients', fontsize=10)
axes[0].set_ylim(0, max(counts.values) + 30)
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 11},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Disease Distribution (%)', fontsize=12, fontweight='bold')

fig.suptitle('Target Variable Distribution', fontsize=14, fontweight='bold', y=1.02)
sns.despine(ax=axes[0])
plt.tight_layout()
plt.savefig('plot1_target_distribution.png', bbox_inches='tight')
plt.show()
print('Insight: The dataset is fairly balanced between heart disease and no heart disease cases.')

In [ ]:
# =====================================================
#  PLOT 2: Age Distribution by Disease Status
# =====================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for val, label, color, ax in zip([0, 1], labels, colors, axes):
    subset = df[df['target'] == val]['age']
    ax.hist(subset, bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(subset.mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'Mean: {subset.mean():.1f} yrs')
    ax.set_title(f'Age Distribution -- {label}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Age (years)', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig.suptitle('Age Distribution by Heart Disease Status', fontsize=14, fontweight='bold', y=1.02)
sns.despine()
plt.tight_layout()
plt.savefig('plot2_age_distribution.png', bbox_inches='tight')
plt.show()
print('Insight: Heart disease patients tend to be older on average.')

In [ ]:
# =====================================================
#  PLOT 3: Gender vs Heart Disease
# =====================================================
fig, ax = plt.subplots(figsize=(7, 4))

sex_disease = df.groupby(['sex', 'target']).size().unstack(fill_value=0)
sex_disease.index = ['Female', 'Male']
sex_disease.columns = ['No Heart Disease', 'Heart Disease']

x = np.arange(len(sex_disease.index))
width = 0.3

bars1 = ax.bar(x - width/2, sex_disease['No Heart Disease'], width,
               label='No Heart Disease', color='#3498DB', edgecolor='white')
bars2 = ax.bar(x + width/2, sex_disease['Heart Disease'], width,
               label='Heart Disease', color='#E74C3C', edgecolor='white')

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')

ax.set_title('Heart Disease Count by Gender', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Gender', fontsize=11)
ax.set_ylabel('Number of Patients', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(['Female', 'Male'], fontsize=11)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
sns.despine()

plt.tight_layout()
plt.savefig('plot3_gender_disease.png', bbox_inches='tight')
plt.show()
print('Insight: Males have a significantly higher incidence of heart disease in this dataset.')

In [ ]:
# =====================================================
#  PLOT 4: Chest Pain Type vs Heart Disease
# =====================================================
fig, ax = plt.subplots(figsize=(9, 4))

cp_labels = {0: 'Typical Angina', 1: 'Atypical Angina',
             2: 'Non-Anginal Pain', 3: 'Asymptomatic'}

cp_disease = df.groupby(['cp', 'target']).size().unstack(fill_value=0)
cp_disease.index = [cp_labels[i] for i in cp_disease.index]
cp_disease.columns = ['No Heart Disease', 'Heart Disease']

x = np.arange(len(cp_disease.index))
width = 0.35

bars1 = ax.bar(x - width/2, cp_disease['No Heart Disease'], width,
               label='No Heart Disease', color='#3498DB', edgecolor='white')
bars2 = ax.bar(x + width/2, cp_disease['Heart Disease'], width,
               label='Heart Disease', color='#E74C3C', edgecolor='white')

ax.set_title('Chest Pain Type vs Heart Disease', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Chest Pain Type', fontsize=11)
ax.set_ylabel('Number of Patients', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(cp_disease.index, rotation=10, fontsize=9)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
sns.despine()

plt.tight_layout()
plt.savefig('plot4_chest_pain.png', bbox_inches='tight')
plt.show()
print('Insight: Asymptomatic chest pain type is most strongly associated with heart disease.')

In [ ]:
# =====================================================
#  PLOT 5: Box Plots -- Key Features by Disease Status
#  (using matplotlib boxplot to avoid seaborn palette issues)
# =====================================================
key_features   = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
feature_labels = ['Age', 'Resting BP', 'Cholesterol', 'Max Heart Rate', 'ST Depression']

fig, axes = plt.subplots(1, 5, figsize=(16, 5))

for i, (feat, label) in enumerate(zip(key_features, feature_labels)):
    no_disease  = df[df['target'] == 0][feat].dropna()
    has_disease = df[df['target'] == 1][feat].dropna()

    bp = axes[i].boxplot(
        [no_disease, has_disease],
        labels=['No\nDisease', 'Heart\nDisease'],
        patch_artist=True,
        medianprops=dict(color='black', linewidth=2),
        flierprops=dict(marker='o', markerfacecolor='gray', markersize=4, alpha=0.5)
    )
    bp['boxes'][0].set_facecolor('#3498DB')
    bp['boxes'][0].set_alpha(0.75)
    bp['boxes'][1].set_facecolor('#E74C3C')
    bp['boxes'][1].set_alpha(0.75)

    axes[i].set_title(label, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Value', fontsize=9)
    axes[i].tick_params(axis='x', labelsize=8)
    axes[i].grid(axis='y', alpha=0.3)

fig.suptitle('Key Clinical Features by Heart Disease Status',
             fontsize=14, fontweight='bold', y=1.02)
sns.despine()
plt.tight_layout()
plt.savefig('plot5_boxplots.png', bbox_inches='tight')
plt.show()
print('Insight: Heart disease patients show lower max heart rate and higher ST depression (oldpeak).')

In [ ]:
# =====================================================
#  PLOT 6: Correlation Heatmap
# =====================================================
fig, ax = plt.subplots(figsize=(11, 8))

corr = df.corr().round(2)

sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.4, linecolor='white',
            annot_kws={'size': 8}, vmin=-1, vmax=1, ax=ax)

ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=12)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig('plot6_heatmap.png', bbox_inches='tight')
plt.show()
print('Insight: thalach has the strongest positive correlation with target.')
print('         oldpeak and exang have strong negative correlations with target.')

In [ ]:
# =====================================================
#  PLOT 7: Histograms of Numeric Features
# =====================================================
numeric_cols  = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
hist_colors   = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, (col, color) in enumerate(zip(numeric_cols, hist_colors)):
    axes[i].hist(df[col], bins=25, color=color, edgecolor='white', alpha=0.85)
    mean_val = df[col].mean()
    axes[i].axvline(mean_val, color='black', linestyle='--', linewidth=1.5,
                    label=f'Mean: {mean_val:.1f}')
    axes[i].set_title(f'Distribution of {col}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(col, fontsize=9)
    axes[i].set_ylabel('Frequency', fontsize=9)
    axes[i].legend(fontsize=9)
    axes[i].grid(alpha=0.3)

axes[5].set_visible(False)
fig.suptitle('Distribution of Numeric Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot7_histograms.png', bbox_inches='tight')
plt.show()

In [ ]:
# =====================================================
#  PLOT 8: Scatter -- Age vs Max Heart Rate
# =====================================================
fig, ax = plt.subplots(figsize=(8, 5))

markers = ['o', '^']
for val, label, color, marker in zip([0, 1], labels, colors, markers):
    subset = df[df['target'] == val]
    ax.scatter(subset['age'], subset['thalach'], label=label,
               color=color, marker=marker, s=60, alpha=0.7,
               edgecolors='white', linewidth=0.4)

ax.set_title('Age vs Maximum Heart Rate by Disease Status',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Age (years)', fontsize=11)
ax.set_ylabel('Max Heart Rate (thalach)', fontsize=11)
ax.legend(title='Status', fontsize=10)
ax.grid(alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig('plot8_scatter_age_thalach.png', bbox_inches='tight')
plt.show()
print('Insight: Patients with heart disease tend to have lower max heart rates.')

---
## Step 6: Data Preprocessing

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

print(f'Features shape: {X.shape}')
print(f'Target shape  : {y.shape}')
print(f'Feature columns: {X.columns.tolist()}')

In [ ]:
# Train-Test Split -- stratified to keep class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train-Test Split (Stratified):')
print(f'  Training samples : {len(X_train)}')
print(f'  Testing  samples : {len(X_test)}')
print(f'  Train disease ratio: {y_train.mean():.2%}')
print(f'  Test  disease ratio: {y_test.mean():.2%}')

In [ ]:
# Feature scaling for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('Feature scaling complete using StandardScaler.')

---
## Step 7: Train Classification Models

In [ ]:
# MODEL 1: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
lr_model.fit(X_train_scaled, y_train)
lr_predictions = lr_model.predict(X_test_scaled)
lr_proba       = lr_model.predict_proba(X_test_scaled)[:, 1]
print('Logistic Regression trained successfully.')

In [ ]:
# MODEL 2: Decision Tree
dt_model = DecisionTreeClassifier(
    max_depth=5, min_samples_split=10,
    min_samples_leaf=5, random_state=42, criterion='gini'
)
dt_model.fit(X_train, y_train)
dt_predictions = dt_model.predict(X_test)
dt_proba       = dt_model.predict_proba(X_test)[:, 1]
print('Decision Tree trained successfully.')
print(f'Tree depth    : {dt_model.get_depth()}')
print(f'Number of leaves: {dt_model.get_n_leaves()}')

---
## Step 8: Model Evaluation

In [ ]:
def evaluate_classifier(name, y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    roc = roc_auc_score(y_true, y_proba)
    print(f'Model         : {name}')
    print(f'Accuracy      : {acc:.4f} ({acc*100:.2f}%)')
    print(f'ROC-AUC Score : {roc:.4f}')
    print()
    print('Classification Report:')
    print(classification_report(y_true, y_pred,
          target_names=['No Disease', 'Heart Disease']))
    return {'Model': name, 'Accuracy': round(acc, 4), 'ROC-AUC': round(roc, 4)}

print('=' * 55)
print('      LOGISTIC REGRESSION RESULTS')
print('=' * 55)
lr_results = evaluate_classifier('Logistic Regression', y_test, lr_predictions, lr_proba)

print('=' * 55)
print('      DECISION TREE RESULTS')
print('=' * 55)
dt_results = evaluate_classifier('Decision Tree', y_test, dt_predictions, dt_proba)

In [ ]:
# Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_cv = cross_val_score(lr_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
dt_cv = cross_val_score(dt_model, X_train,        y_train, cv=cv, scoring='accuracy')

print('5-Fold Cross Validation Results:')
print('-' * 45)
print(f'Logistic Regression -- Mean: {lr_cv.mean():.4f}, Std: {lr_cv.std():.4f}')
print(f'  Fold scores: {[round(s,3) for s in lr_cv]}')
print()
print(f'Decision Tree       -- Mean: {dt_cv.mean():.4f}, Std: {dt_cv.std():.4f}')
print(f'  Fold scores: {[round(s,3) for s in dt_cv]}')

---
## Step 9: Visualizations -- Model Performance

In [ ]:
# =====================================================
#  PLOT 9: Confusion Matrices
# =====================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, preds, name, cmap in zip(
        axes,
        [lr_predictions, dt_predictions],
        ['Logistic Regression', 'Decision Tree'],
        ['Blues', 'Greens']):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['No Disease', 'Heart Disease']
    )
    disp.plot(ax=ax, cmap=cmap, colorbar=False)
    ax.set_title(f'Confusion Matrix -- {name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('True Label', fontsize=10)

fig.suptitle('Confusion Matrix Comparison', fontsize=14, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig('plot9_confusion_matrix.png', bbox_inches='tight')
plt.show()
print('Insight: Diagonal cells are correct predictions. Off-diagonal are errors.')

In [ ]:
# =====================================================
#  PLOT 10: ROC Curve -- Both Models
# =====================================================
fig, ax = plt.subplots(figsize=(7, 5))

model_data = [
    ('Logistic Regression', lr_proba, '#E74C3C'),
    ('Decision Tree',       dt_proba, '#27AE60')
]

for name, proba, color in model_data:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC = 0.500)')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_title('ROC Curve Comparison', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.legend(fontsize=10, loc='lower right')
ax.grid(alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig('plot10_roc_curve.png', bbox_inches='tight')
plt.show()
print('Insight: AUC closer to 1.0 means better model. Both models beat random guessing significantly.')

In [ ]:
# =====================================================
#  PLOT 11: Feature Importance -- Logistic Regression Coefficients
# =====================================================
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))

bar_colors = ['#E74C3C' if c < 0 else '#27AE60' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'],
        color=bar_colors, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Logistic Regression -- Feature Coefficients\n(Green = increases risk, Red = decreases risk)',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Coefficient Value', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
ax.grid(axis='x', alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig('plot11_lr_coefficients.png', bbox_inches='tight')
plt.show()
print('Insight: Features with large positive coefficients strongly predict heart disease.')

In [ ]:
# =====================================================
#  PLOT 12: Feature Importance -- Decision Tree
# =====================================================
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))

colors_imp = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(importance_df)))
bars = ax.barh(importance_df['Feature'], importance_df['Importance'],
               color=colors_imp, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, importance_df['Importance']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_title('Decision Tree -- Feature Importance', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Importance Score', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
ax.grid(axis='x', alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig('plot12_dt_feature_importance.png', bbox_inches='tight')
plt.show()
print('Insight: Higher importance score = more influence on heart disease prediction.')

In [ ]:
# =====================================================
#  PLOT 13: Decision Tree Visualization
# =====================================================
fig, ax = plt.subplots(figsize=(20, 8))

plot_tree(
    dt_model,
    feature_names=X.columns.tolist(),
    class_names=['No Disease', 'Heart Disease'],
    filled=True, rounded=True, max_depth=3,
    ax=ax, fontsize=9, impurity=False
)
ax.set_title('Decision Tree Visualization (Top 3 Levels)', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('plot13_decision_tree.png', bbox_inches='tight', dpi=150)
plt.show()
print('Insight: Each node shows the feature and threshold used to split patients.')

In [ ]:
# =====================================================
#  PLOT 14: Accuracy and ROC-AUC Bar Comparison
# =====================================================
model_names = ['Logistic Regression', 'Decision Tree']
accuracies  = [lr_results['Accuracy'], dt_results['Accuracy']]
roc_aucs    = [lr_results['ROC-AUC'], dt_results['ROC-AUC']]

x = np.arange(len(model_names))
width = 0.3

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy',
               color='#3498DB', edgecolor='white')
bars2 = ax.bar(x + width/2, roc_aucs, width, label='ROC-AUC',
               color='#E74C3C', edgecolor='white')

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom',
            fontsize=10, fontweight='bold')

ax.set_ylim(0.5, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig('plot14_model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# =====================================================
#  PLOT 15: Cross Validation Scores per Fold
# =====================================================
fig, ax = plt.subplots(figsize=(8, 4))
folds = [f'Fold {i+1}' for i in range(5)]

ax.plot(folds, lr_cv, 'o-', color='#E74C3C', linewidth=2, markersize=7,
        label=f'Logistic Regression (Mean={lr_cv.mean():.3f})')
ax.plot(folds, dt_cv, 's-', color='#27AE60', linewidth=2, markersize=7,
        label=f'Decision Tree (Mean={dt_cv.mean():.3f})')

ax.set_title('5-Fold Cross Validation Accuracy per Fold',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Fold', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_ylim(0.6, 1.0)
ax.legend(fontsize=10)
ax.grid(alpha=0.4)
sns.despine()
plt.tight_layout()
plt.savefig('plot15_cross_validation.png', bbox_inches='tight')
plt.show()
print('Insight: Consistent accuracy across folds means the model generalizes well.')

---
## Step 10: Final Summary

In [ ]:
summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'Test Accuracy': [lr_results['Accuracy'], dt_results['Accuracy']],
    'ROC-AUC': [lr_results['ROC-AUC'], dt_results['ROC-AUC']],
    'CV Mean Accuracy': [round(lr_cv.mean(), 4), round(dt_cv.mean(), 4)],
    'CV Std': [round(lr_cv.std(), 4), round(dt_cv.std(), 4)]
}).set_index('Model')

print('Final Model Comparison Summary:')
print('=' * 60)
summary

In [ ]:
top_features = importance_df.sort_values('Importance', ascending=False).head(5)
print('Top 5 Most Important Features:')
print('=' * 45)
for i, (_, row) in enumerate(top_features.iterrows(), 1):
    print(f'  {i}. {row["Feature"]:12s} -- Importance: {row["Importance"]:.4f}')

---
## Step 11: Conclusion and Insights

### Key Findings

| # | Finding |
|---|---|
| 1 | The dataset has no missing values and is fairly balanced, making it suitable for direct modeling |
| 2 | Males have a significantly higher rate of heart disease than females in this dataset |
| 3 | Asymptomatic chest pain type is most strongly associated with heart disease |
| 4 | Patients with heart disease tend to have lower maximum heart rate (thalach) |
| 5 | Higher ST depression (oldpeak) is a strong indicator of heart disease |
| 6 | thalach, cp, ca, thal, and oldpeak are the most predictive features |
| 7 | Both models achieve over 80% accuracy with ROC-AUC above 0.85 |
| 8 | Logistic Regression provides interpretable coefficients useful in a clinical setting |

### Conclusion
This task demonstrated how binary classification models can be applied to medical data to predict heart disease. By combining preprocessing, EDA, model training, and evaluation with metrics like ROC-AUC and confusion matrix, we built models that can assist in early detection of heart disease. Logistic Regression is recommended for clinical use due to its interpretability.

---
### Tools and Libraries Used
- pandas, numpy -- Data loading and manipulation
- scikit-learn -- Logistic Regression, Decision Tree, evaluation metrics
- matplotlib, seaborn -- Visualization

---
*DevelopersHub Corporation -- AI/ML Engineering Internship | Task 3*